In [21]:
import pandas as pd
import numpy as np
import random
import itertools

In [22]:
from models.classification.RandomForestClassifier import RFC
from models.classification.XGBClassifier import XGBC
from models.classification.LR import LRWrapper
from models.classification.SVC import SVCWrapper

In [68]:
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import f1_score, accuracy_score

In [24]:
from utilities import get_predictors_basic

In [25]:
from tqdm import tqdm

#### Setup grids for hyperparameter tuning

In [92]:
param_grid_rf = {
    "n_estimators": [400, 800, 1200, 1600, 2000, 2400],
    "max_depth": [None, 10, 20, 40],
    "min_samples_split": [2, 5, 10, 25, 50, 100, 150],
    "min_samples_leaf": [1, 2, 4, 6, 8, 10],
    "max_features": ["sqrt", "log2"],
    "bootstrap": [True, False],
    "class_weight": ["balanced"]
}
param_grid_xgb = {
    "n_estimators": [400, 800, 1200, 1600, 2000, 2400],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "max_depth": [3, 5, 7, 9],
    "min_child_weight": [1, 3, 5],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "gamma": [0, 0.1, 0.3],
    "reg_lambda": [1, 1.5, 2],
    "reg_alpha": [0, 0.1, 0.5]
}
param_grid_lr = {
    'penalty': ['l2', None],  # Regularization type
    'C': [0.01, 0.1, 1, 10, 100],  # Regularization strength (inverse of regularization)
    'solver': ['sag', 'newton-cg', 'lbfgs'],  # Algorithm to use
    'max_iter': [200, 400, 600, 800, 1000],  # Maximum number of iterations
    'tol': [1e-4, 1e-3, 1e-2],  # Tolerance for stopping criteria
    'class_weight': ['balanced'],  # Class weight
    'fit_intercept': [True, False],  # Whether to include intercept
    'warm_start': [True, False],  # Reuse the solution of the previous call to fit
}
param_grid_svc = {
    'C': [0.01, 0.1, 1, 10, 100],  # Regularization parameter
    'kernel': ['linear', 'poly', 'rbf', 'sigmoid'],  # Kernel type
    'degree': [2, 3, 4],  # Degree of the polynomial kernel function (only used for 'poly')
    'gamma': ['scale', 'auto', 0.001, 0.01, 0.1],  # Kernel coefficient (for 'rbf', 'poly', 'sigmoid')
    'coef0': [0.0, 0.1, 0.5, 1],  # Independent term in kernel function (only used for 'poly' and 'sigmoid')
    'shrinking': [True, False],  # Whether to use the shrinking heuristic
    'class_weight': ['balanced'],  # Class weight
    'tol': [1e-4, 1e-3, 1e-2],  # Tolerance for stopping criteria
    'max_iter': [1000, 2000, 3000],  # Maximum number of iterations
}


In [93]:
param_grids = [param_grid_rf, param_grid_xgb, param_grid_lr, param_grid_lr, param_grid_svc, param_grid_svc]

In [94]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

#### Import data

In [95]:
matches_df = pd.read_csv('data/processed/process_ENG1.csv')

In [96]:
matches_df["xg_rolling_diff"] = abs(matches_df["xg_rolling_forHomeTeam"] - matches_df["xg_rolling_forAwayTeam"])
matches_df["xg_mean_diff"] = abs(matches_df["xg_mean_forHomeTeam"] - matches_df["xg_mean_forAwayTeam"])
matches_df["xa_rolling_diff"] = abs(matches_df["xa_rolling_forHomeTeam"] - matches_df["xa_rolling_forAwayTeam"])
matches_df["xa_mean_diff"] = abs(matches_df["xa_mean_forHomeTeam"] - matches_df["xa_mean_forAwayTeam"])
matches_df["xga_rolling_diff"] = abs(matches_df["xga_rolling_forHomeTeam"] - matches_df["xga_rolling_forAwayTeam"])
matches_df["xga_mean_diff"] = abs(matches_df["xga_mean_forHomeTeam"] - matches_df["xga_mean_forAwayTeam"])
matches_df["gf_rolling_diff"] = abs(matches_df["gf_rolling_forHomeTeam"] - matches_df["gf_rolling_forAwayTeam"])
matches_df["gf_mean_diff"] = abs(matches_df["gf_mean_forHomeTeam"] - matches_df["gf_mean_forAwayTeam"])
matches_df["ga_rolling_diff"] = abs(matches_df["ga_rolling_forHomeTeam"] - matches_df["ga_rolling_forAwayTeam"])
matches_df["ga_mean_diff"] = abs(matches_df["ga_mean_forHomeTeam"] - matches_df["ga_mean_forAwayTeam"])

In [97]:
matches_df_train, matches_df_test = train_test_split(matches_df, test_size=0.25, random_state=42, stratify=matches_df['result_code'])

In [98]:
model_names = [
    "RFC", "XGBC", "LogisticRegression_1vR", "LogisticRegression_1v1", "SVC_1vR", "SVC_1v1"
]

In [99]:
predictors = get_predictors_basic() + [
    "xg_rolling_diff",
    "xg_mean_diff",
    "xa_rolling_diff",
    "xa_mean_diff",
    "xga_rolling_diff",
    "xga_mean_diff",
    "gf_rolling_diff",
    "gf_mean_diff",
    "ga_rolling_diff",
    "ga_mean_diff"
]

In [100]:
random.seed(42)
MIN_COMBINATIONS = 200

In [101]:
best_params = []
best_f1 = []
accuracies = []

for model, param_grid in tqdm(zip(model_names, param_grids), total=len(model_names), desc="Models"):
    best_model_params = None
    best_model_score = -np.inf
    accuracy = 0

    all_combinations = list(itertools.product(*param_grid.values()))

    sampled_combinations = random.sample(all_combinations, min(MIN_COMBINATIONS, len(all_combinations)))

    for combo in tqdm(sampled_combinations, desc=f"Tuning {model}"):
        params = dict(zip(param_grid.keys(), combo))
        
        if model == "RFC":
            clf = RFC(params)
        elif model == "XGBC":
            clf = XGBC(params)
        elif model == "LogisticRegression_1vR":
            clf = LRWrapper(params, one_vs_rest=True)
        elif model == "LogisticRegression_1v1":
            clf = LRWrapper(params, one_vs_rest=False)
        elif model == "SVC_1vR":
            clf = SVCWrapper(params, one_vs_rest=True)
        elif model == "SVC_1v1":
            clf = SVCWrapper(params, one_vs_rest=False)
        else:
            continue  # Unknown model, skip

        clf.train(matches_df_train, get_predictors_basic())

        preds = clf.evaluate_model(matches_df_test, get_predictors_basic())

        # Calculate F1 score
        f1 = f1_score(preds["Actual_Result"], preds["Predicted_Result"], average='weighted')

        # Update best parameters if current F1 is better
        if f1 > best_model_score:
            best_model_score = f1
            best_model_params = params
            accuracy = accuracy_score(preds["Actual_Result"], preds["Predicted_Result"])

            
    best_params.append(best_model_params)
    best_f1.append(best_model_score)
    accuracies.append(accuracy)


Models:  33%|███▎      | 2/6 [25:07<48:41, 730.37s/it]  c:\Users\negia\Desktop\Programming\Match-Predictor-FBREF\match-predictor-venv\lib\site-packages\sklearn\linear_model\_logistic.py:1181: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(
c:\Users\negia\Desktop\Programming\Match-Predictor-FBREF\match-predictor-venv\lib\site-packages\sklearn\linear_model\_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\negia\Desktop\Programming\Match-Predictor-FBREF\match-predictor-venv\lib\site-packages\sklearn\linear_model\_logistic.py:1181: UserWarning:

In [102]:
best_f1

[0.5439524180900388,
 0.5430578263178233,
 0.5453479670740833,
 0.5595633377794174,
 0.5527725118483413,
 0.5468847704735378]

In [103]:
accuracies

[0.5639810426540285,
 0.5592417061611374,
 0.5497630331753555,
 0.5545023696682464,
 0.5545023696682464,
 0.5355450236966824]

In [104]:
best_params

[{'n_estimators': 1200,
  'max_depth': 10,
  'min_samples_split': 5,
  'min_samples_leaf': 10,
  'max_features': 'sqrt',
  'bootstrap': True,
  'class_weight': 'balanced'},
 {'n_estimators': 2400,
  'learning_rate': 0.2,
  'max_depth': 9,
  'min_child_weight': 3,
  'subsample': 0.6,
  'colsample_bytree': 0.6,
  'gamma': 0.1,
  'reg_lambda': 2,
  'reg_alpha': 0.1},
 {'penalty': 'l2',
  'C': 0.01,
  'solver': 'sag',
  'max_iter': 200,
  'tol': 0.001,
  'class_weight': 'balanced',
  'fit_intercept': False,
  'warm_start': False},
 {'penalty': 'l2',
  'C': 100,
  'solver': 'lbfgs',
  'max_iter': 200,
  'tol': 0.01,
  'class_weight': 'balanced',
  'fit_intercept': True,
  'warm_start': True},
 {'C': 100,
  'kernel': 'rbf',
  'degree': 3,
  'gamma': 'scale',
  'coef0': 0.5,
  'shrinking': False,
  'class_weight': 'balanced',
  'tol': 0.01,
  'max_iter': 3000},
 {'C': 0.1,
  'kernel': 'rbf',
  'degree': 4,
  'gamma': 0.001,
  'coef0': 0.1,
  'shrinking': True,
  'class_weight': 'balanced',
  